# QDR-AttNet — final Kaggle-ready implementation

**Quantum Data Re-uploading Attention Network** for multiclass plant-leaf-disease classification.

**Execution order:** install dependencies → prepare/download the Kaggle dataset → configure the model → extract/cache features → run QDR-AttNet.

The Kaggle dataset is therefore resolved **before** any model/training code is executed.


## 0. Install dependencies
Run once (uncomment). `torch` from PyPI is the main download; CUDA is used automatically if present.

In [ ]:
# Auto-install required packages if they are missing.
# Safe for Kaggle / Colab / Jupyter. Run this cell first.
import importlib.util
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "PIL": "Pillow",
    "torch": "torch",
    "torchvision": "torchvision",
    "pennylane": "pennylane",
    "kagglehub": "kagglehub",
}

missing = [pip_name for module, pip_name in REQUIRED.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

# Verify the imports needed by the notebook.
import numpy as np
import torch
import torchvision
import pennylane as qml

print("NumPy     :", np.__version__)
print("PyTorch   :", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PennyLane :", qml.__version__)


Installing missing packages: pennylane
NumPy     : 2.1.3
PyTorch   : 2.11.0+cpu
Torchvision: 0.26.0+cpu
PennyLane : 0.45.1


## 1. Dataset utilities
These helpers locate and validate the 38-class ImageFolder root and can download the public Kaggle dataset when required.


In [ ]:
import os
import random
import subprocess
import sys
import numpy as np
import torch


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(pref="cuda"):
    return torch.device("cuda" if (pref == "cuda" and torch.cuda.is_available()) else "cpu")


def ensure_dir(p):
    if p:
        os.makedirs(p, exist_ok=True)


IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff")


def _looks_like_class_dir(path):
    """True when `path` directly contains at least one image file."""
    try:
        with os.scandir(path) as it:
            return any(
                e.is_file() and e.name.lower().endswith(IMG_EXTS)
                for e in it
            )
    except OSError:
        return False


def _class_dirs(path):
    """Return class-like child directories (children that directly contain images)."""
    if not path or not os.path.isdir(path):
        return []
    try:
        return sorted(
            d for d in os.listdir(path)
            if os.path.isdir(os.path.join(path, d))
            and _looks_like_class_dir(os.path.join(path, d))
        )
    except OSError:
        return []


def is_valid_image_root(path, expected_classes=38):
    """Validate that path directly contains the expected number of class folders."""
    return len(_class_dirs(path)) == expected_classes


def find_image_root(search_roots=None, expected_classes=38, max_depth=10,
                    prefer_name="train"):
    """Locate the directory that directly contains the class folders.

    This handles the doubly nested layout of the Kaggle New Plant Diseases Dataset.
    Exact expected-class matches are preferred, and a folder named `train` receives
    the highest preference.
    """
    if search_roots is None:
        search_roots = [
            "/kaggle/input",
            "/kaggle/working",
            "/content",
            "/root/.cache/kagglehub",
            "data",
            ".",
        ]

    candidates = []

    for root in search_roots:
        root = os.path.abspath(os.path.expanduser(str(root)))
        if not os.path.isdir(root):
            continue

        root_depth = root.rstrip(os.sep).count(os.sep)

        for dirpath, dirnames, _ in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]

            depth = dirpath.rstrip(os.sep).count(os.sep) - root_depth
            if depth > max_depth:
                dirnames[:] = []
                continue

            if not dirnames:
                continue

            n_class = len(_class_dirs(dirpath))
            if n_class == 0:
                continue

            # Strongly prefer exact 38-class roots and then the `train` directory.
            score = -abs(n_class - expected_classes) * 100
            if n_class == expected_classes:
                score += 10000
            if os.path.basename(dirpath).lower() == prefer_name.lower():
                score += 1000

            candidates.append((score, n_class, dirpath))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    best = candidates[0]

    # Do not silently accept a wrong class count.
    if best[1] != expected_classes:
        return None

    return best[2]


def _install_kagglehub_if_needed():
    try:
        import kagglehub
        return kagglehub
    except ModuleNotFoundError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "kagglehub"
        ])
        import kagglehub
        return kagglehub


def download_plant_disease_dataset(verbose=True):
    """Download the Kaggle dataset with kagglehub and return its local base path.

    Internet/Kaggle access is required when the dataset has not already been attached.
    """
    kagglehub = _install_kagglehub_if_needed()
    if verbose:
        print("Dataset not found locally. Trying KaggleHub download...")

    return kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")


def resolve_data_root(cfg, verbose=True, allow_download=False):
    """Resolve and validate cfg.data_root.

    Order:
      1. use cfg.data_root if it is already a valid 38-class ImageFolder root;
      2. search standard Kaggle/Colab/local/cache locations;
      3. optionally download via KaggleHub only when allow_download=True.

    The final notebook prepares the dataset before Config is created, so later calls
    normally use allow_download=False.
    """
    if is_valid_image_root(cfg.data_root, cfg.num_classes):
        if verbose:
            print("Using configured data_root:", cfg.data_root)
        return cfg.data_root

    standard_kaggle_path = (
        "/kaggle/input/new-plant-diseases-dataset/"
        "New Plant Diseases Dataset(Augmented)/"
        "New Plant Diseases Dataset(Augmented)/train"
    )

    search_roots = [
        standard_kaggle_path,
        "/kaggle/input",
        "/kaggle/working",
        "/content",
        "/root/.cache/kagglehub",
        "data",
        ".",
    ]

    found = find_image_root(
        search_roots=search_roots,
        expected_classes=cfg.num_classes,
        max_depth=10,
        prefer_name="train",
    )

    if found is None and allow_download:
        try:
            downloaded = download_plant_disease_dataset(verbose=verbose)
            found = find_image_root(
                search_roots=[downloaded],
                expected_classes=cfg.num_classes,
                max_depth=12,
                prefer_name="train",
            )
        except Exception as e:
            if verbose:
                print("Automatic Kaggle download failed:", repr(e))

    if found is None:
        raise FileNotFoundError(
            "Could not locate the 38-class New Plant Diseases Dataset.\n\n"
            "On Kaggle, click Add Input and add:\n"
            "  vipoooool/new-plant-diseases-dataset\n\n"
            "Then rerun this cell. The expected train path is usually:\n"
            "/kaggle/input/new-plant-diseases-dataset/"
            "New Plant Diseases Dataset(Augmented)/"
            "New Plant Diseases Dataset(Augmented)/train\n\n"
            "Alternatively set CFG.data_root manually to the directory that directly "
            f"contains all {cfg.num_classes} class folders."
        )

    cfg.data_root = found
    classes = _class_dirs(found)

    if verbose:
        print("Resolved data_root:", cfg.data_root)
        print("Number of classes:", len(classes))
        print("First classes:", classes[:5])

    return cfg.data_root


## 2. Prepare the Kaggle dataset first
This cell runs immediately after dependency installation. It uses the Kaggle input copy when already attached; otherwise it downloads `vipoooool/new-plant-diseases-dataset` with KaggleHub, then resolves the 38-class `train` directory.


In [ ]:
# ==================== DATASET PREPARATION: RUN BEFORE MODEL CODE ====================
import os

EXPECTED_CLASSES = 38
KAGGLE_DATASET_SLUG = "vipoooool/new-plant-diseases-dataset"
STANDARD_KAGGLE_TRAIN = (
    "/kaggle/input/new-plant-diseases-dataset/"
    "New Plant Diseases Dataset(Augmented)/"
    "New Plant Diseases Dataset(Augmented)/train"
)


def prepare_dataset_first():
    """Return the 38-class training root before the rest of QDR-AttNet is defined.

    Priority:
      1) use the dataset already attached to a Kaggle notebook;
      2) use an already cached/local copy if one exists;
      3) download the public Kaggle dataset with KaggleHub;
      4) locate and validate the directory containing exactly 38 class folders.
    """
    # Fast path: Kaggle Add Input already provides the dataset.
    if is_valid_image_root(STANDARD_KAGGLE_TRAIN, EXPECTED_CLASSES):
        print("Using Kaggle input dataset:", STANDARD_KAGGLE_TRAIN)
        return STANDARD_KAGGLE_TRAIN

    # Reuse an existing local/cache copy before downloading again.
    found = find_image_root(
        search_roots=[
            "/kaggle/input",
            "/kaggle/working",
            "/content",
            "/root/.cache/kagglehub",
            "data",
            ".",
        ],
        expected_classes=EXPECTED_CLASSES,
        max_depth=10,
        prefer_name="train",
    )
    if found is not None:
        print("Using existing dataset:", found)
        return found

    # No usable copy exists: download NOW, before configuration/model/training.
    print(f"Dataset not found locally. Downloading {KAGGLE_DATASET_SLUG} ...")
    downloaded_base = download_plant_disease_dataset(verbose=False)
    print("KaggleHub dataset base:", downloaded_base)

    # Check the common folder layouts first.
    common_candidates = [
        os.path.join(
            downloaded_base,
            "New Plant Diseases Dataset(Augmented)",
            "New Plant Diseases Dataset(Augmented)",
            "train",
        ),
        os.path.join(
            downloaded_base,
            "New Plant Diseases Dataset(Augmented)",
            "train",
        ),
        os.path.join(downloaded_base, "train"),
    ]

    for candidate in common_candidates:
        if is_valid_image_root(candidate, EXPECTED_CLASSES):
            print("Downloaded dataset resolved to:", candidate)
            return candidate

    # Fallback for any future Kaggle directory-layout changes.
    found = find_image_root(
        search_roots=[downloaded_base],
        expected_classes=EXPECTED_CLASSES,
        max_depth=12,
        prefer_name="train",
    )
    if found is None:
        raise FileNotFoundError(
            "The Kaggle dataset was downloaded but the directory containing exactly "
            f"{EXPECTED_CLASSES} class folders could not be found under: {downloaded_base}"
        )

    print("Downloaded dataset resolved to:", found)
    return found


DATA_ROOT = prepare_dataset_first()

print("\nDataset preparation completed BEFORE model setup.")
print("DATA_ROOT:", DATA_ROOT)
print("Classes  :", len(_class_dirs(DATA_ROOT)))


Dataset not found locally. Downloading vipoooool/new-plant-diseases-dataset ...
Using Colab cache for faster access to the 'new-plant-diseases-dataset' dataset.
KaggleHub dataset base: /kaggle/input/new-plant-diseases-dataset
Downloaded dataset resolved to: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train

Dataset preparation completed BEFORE model setup.
DATA_ROOT: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Classes  : 38


## 3. Configuration
The dataset path below comes from the dataset-preparation cell above; the remaining hyperparameters preserve the original implementation.


In [ ]:
"""Central configuration for QDR-AttNet. Values mirror the paper's Table settings."""
from dataclasses import dataclass


@dataclass
class Config:
    # ---- data ----
    # Point this to an ImageFolder root with 38 class sub-directories.
    # For the New Plant Diseases Dataset, merge/point to the folder that holds the
    # 38 class folders (train and valid share identical class names).
    data_root: str = DATA_ROOT
    cache_path: str = "cache/features.npz"
    num_classes: int = 38
    image_size: int = 224

    # ---- backbone dims ----
    effnet_dim: int = 1280          # EfficientNetV2-S GAP dim
    convnext_dim: int = 768         # ConvNeXt-Tiny GAP dim
    proj_dim: int = 512             # projection of ConvNeXt features
    fused_dim: int = 1280 + 512     # 1792

    # ---- QMI-attention front-end ----
    attn_reduction: int = 16
    frontend_epochs: int = 15
    frontend_lr: float = 1e-3
    redundancy_subset: int = 256    # random channels used for the differentiable redundancy term
    beta_redundancy: float = 0.3
    gate_l1: float = 1e-4

    # ---- CAPSO feature selection ----
    k_features: int = 6             # qubit-aligned subset size
    swarm_size: int = 40
    capso_iters: int = 60
    w_max: float = 0.9
    w_min: float = 0.4
    c1: float = 1.494
    c2: float = 1.494
    lam_redundancy: float = 0.3
    capso_subsample: int = 4000     # rows used for fast fitness evaluation

    # ---- data re-uploading quantum layer ----
    n_qubits: int = 6
    reupload_depth: int = 3         # D

    # ---- hybrid head / optimization ----
    head_hidden: int = 512
    dropout: float = 0.4
    weight_decay: float = 1e-3
    gamma_sep: float = 0.1          # quantum-separability regularizer weight
    lr: float = 1e-3
    max_epochs: int = 50
    patience: int = 10
    batch_size: int = 64

    # ---- cross-validation ----
    n_folds: int = 5
    seeds: tuple = (42, 43, 44)
    val_frac: float = 0.10
    ece_bins: int = 15             # bins for Expected Calibration Error (Fig. 12)
    device: str = "cuda"            # falls back to cpu automatically
    out_dir: str = "results"


CFG = Config()


## 4. Preprocessing & augmentation (Section III-A)

In [ ]:
"""Dataset construction, preprocessing, and augmentation (Section III-A of the paper)."""
from torchvision import transforms, datasets

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def eval_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def train_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.RandomResizedCrop(size, scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


def build_dataset(root, train=False, size=224):
    tf = train_transform(size) if train else eval_transform(size)
    return datasets.ImageFolder(root, transform=tf)

## 5. Frozen dual-path backbone (Section III-B)
EfficientNetV2-S (1280-D) + ConvNeXt-Tiny (768-D).

In [ ]:
"""Frozen dual-path backbone: EfficientNetV2-S + ConvNeXt-Tiny (Section III-B)."""
import torch
import torch.nn as nn
from torchvision.models import (
    efficientnet_v2_s, EfficientNet_V2_S_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
)


class DualBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        eff = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        cvx = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        self.eff_features = eff.features
        self.cvx_features = cvx.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, x):
        u = self.pool(self.eff_features(x)).flatten(1)   # (B, 1280)
        v = self.pool(self.cvx_features(x)).flatten(1)   # (B, 768)
        return u, v

## 6. Feature extraction / caching
Backbones are frozen, so embeddings `(u, v)` are computed once and cached.

In [ ]:
"""Precompute and cache frozen backbone embeddings (u, v) for the whole dataset."""
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


def extract(root=None, cache_path=None, batch_size=64, device=None):
    root = root or resolve_data_root(CFG, allow_download=False)
    if not is_valid_image_root(root, CFG.num_classes):
        # If a parent/nested path was supplied, resolve the actual ImageFolder root.
        CFG.data_root = root
        root = resolve_data_root(CFG, allow_download=False)

    cache_path = cache_path or CFG.cache_path
    device = device or get_device(CFG.device)

    print("Feature extraction dataset:", root)
    print("Device:", device)

    ds = build_dataset(root, train=False, size=CFG.image_size)
    if len(ds.classes) != CFG.num_classes:
        raise ValueError(
            f"Expected {CFG.num_classes} classes but ImageFolder found {len(ds.classes)} "
            f"in {root}."
        )

    # pin_memory helps GPU input transfer; 2 workers is safer across hosted notebooks.
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(device.type == "cuda"),
    )

    bb = DualBackbone().to(device)
    U, V, Y = [], [], []

    for x, y in tqdm(dl, desc="extract features"):
        x = x.to(device, non_blocking=True)
        u, v = bb(x)
        U.append(u.cpu().numpy())
        V.append(v.cpu().numpy())
        Y.append(y.numpy())

    U = np.concatenate(U)
    V = np.concatenate(V)
    Y = np.concatenate(Y)

    ensure_dir(os.path.dirname(cache_path))
    np.savez_compressed(
        cache_path,
        u=U,
        v=V,
        y=Y,
        classes=np.array(ds.classes),
        data_root=np.array([os.path.abspath(root)]),
    )

    print("Saved cache:", cache_path)
    print("U:", U.shape, "V:", V.shape, "Y:", Y.shape)
    return U, V, Y


def load_or_extract(cache_path=None, force_reextract=False, **kw):
    cache_path = cache_path or CFG.cache_path

    if os.path.exists(cache_path) and not force_reextract:
        d = np.load(cache_path, allow_pickle=True)
        print("Loaded cached backbone features:", cache_path)
        print("U:", d["u"].shape, "V:", d["v"].shape, "Y:", d["y"].shape)
        return d["u"], d["v"], d["y"]

    return extract(cache_path=cache_path, **kw)


## 7. QMI-Attention feature distillation (Section III-C)
Squeeze-excitation gating trained with a cross-entropy relevance term + a differentiable redundancy penalty.

In [ ]:
"""QMI-Attention feature distillation and phase-A front-end training (Section III-C)."""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F



class FrontEnd(nn.Module):
    """v-projection + squeeze-excitation channel attention (+ linear probe for phase A)."""

    def __init__(self, cfg=CFG):
        super().__init__()
        self.proj = nn.Linear(cfg.convnext_dim, cfg.proj_dim)
        d = cfg.fused_dim
        r = max(1, d // cfg.attn_reduction)
        self.gate = nn.Sequential(
            nn.Linear(d, r), nn.ReLU(inplace=True),
            nn.Linear(r, d), nn.Sigmoid(),
        )
        self.probe = nn.Linear(d, cfg.num_classes)  # used only during phase-A training
        self.register_buffer("fmean", torch.zeros(d))
        self.register_buffer("fstd", torch.ones(d))

    def fuse(self, u, v):
        return torch.cat([u, self.proj(v)], dim=1)          # (B, 1792)

    def refine(self, u, v):
        z = self.fuse(u, v)
        return z * self.gate(z)                              # gated distillation

    def forward(self, u, v):
        zr = self.refine(u, v)
        return self.probe(zr), zr


def redundancy_penalty(feats, subset, device):
    """Differentiable mean absolute pairwise Pearson correlation on a random channel subset."""
    idx = torch.randperm(feats.shape[1], device=device)[:subset]
    x = feats[:, idx]
    x = x - x.mean(0, keepdim=True)
    x = x / (x.std(0, keepdim=True) + 1e-6)
    c = (x.T @ x) / (x.shape[0] - 1)
    n = c.shape[0]
    off = c.abs().sum() - torch.diagonal(c).abs().sum()
    return off / (n * (n - 1) + 1e-9)


def train_frontend(u, v, y, cfg=CFG, device=None, epochs=None, verbose=True):
    device = device or torch.device("cpu")
    epochs = epochs or cfg.frontend_epochs
    fe = FrontEnd(cfg).to(device)
    u_t = torch.as_tensor(u, dtype=torch.float32, device=device)
    v_t = torch.as_tensor(v, dtype=torch.float32, device=device)
    y_t = torch.as_tensor(y, dtype=torch.long, device=device)
    opt = torch.optim.Adam(fe.parameters(), lr=cfg.frontend_lr, weight_decay=1e-4)
    n, bs = u_t.shape[0], 256
    for ep in range(epochs):
        perm = torch.randperm(n, device=device)
        tot = 0.0
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            logits, zr = fe(u_t[b], v_t[b])
            ce = F.cross_entropy(logits, y_t[b])                        # relevance proxy
            red = redundancy_penalty(zr, cfg.redundancy_subset, device)  # redundancy penalty
            l1 = fe.gate(fe.fuse(u_t[b], v_t[b])).mean()                 # gate sparsity
            loss = ce + cfg.beta_redundancy * red + cfg.gate_l1 * l1
            opt.zero_grad()
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
        if verbose:
            print(f"[front-end] epoch {ep + 1}/{epochs} loss {tot / n:.4f}")
    with torch.no_grad():
        zr = fe.refine(u_t, v_t)
        fe.fmean.copy_(zr.mean(0))
        fe.fstd.copy_(zr.std(0) + 1e-6)
    return fe


@torch.no_grad()
def compute_refined(fe, u, v, device=None):
    device = device or next(fe.parameters()).device
    u_t = torch.as_tensor(u, dtype=torch.float32, device=device)
    v_t = torch.as_tensor(v, dtype=torch.float32, device=device)
    out = []
    for i in range(0, u_t.shape[0], 1024):
        out.append(fe.refine(u_t[i:i + 1024], v_t[i:i + 1024]).cpu().numpy())
    return np.concatenate(out)

## 8. CAPSO qubit-aligned selection (Section III-D)
Chaotic adaptive PSO with a Random-Forest prior and a hard top-k=6 cardinality constraint.

In [ ]:
"""Chaotic Adaptive PSO with Random-Forest prior for qubit-aligned selection (Section III-D)."""
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import f1_score



def _redundancy(X):
    if X.shape[1] < 2:
        return 0.0
    c = np.corrcoef(X, rowvar=False)
    n = c.shape[0]
    off = np.abs(c).sum() - np.abs(np.diag(c)).sum()
    return off / (n * (n - 1))


def _fitness(idx, Xtr, ytr, Xval, yval, lam):
    if len(idx) == 0:
        return -1e9
    clf = LinearDiscriminantAnalysis()
    try:
        clf.fit(Xtr[:, idx], ytr)
        f1 = f1_score(yval, clf.predict(Xval[:, idx]), average="macro", zero_division=0)
    except Exception:
        return -1e9
    return f1 - lam * _redundancy(Xtr[:, idx])


def _topk_mask(pos, k):
    p = 1.0 / (1.0 + np.exp(-pos))     # sigmoid -> selection probabilities
    return np.argsort(-p)[:k]          # deterministic top-k


def capso_select(X, y, cfg=CFG, seed=0, verbose=True):
    """Return the indices of k discriminative, low-redundancy features and the fitness history."""
    rng = np.random.default_rng(seed)
    N, D = X.shape

    if N > cfg.capso_subsample:
        sub = rng.choice(N, cfg.capso_subsample, replace=False)
        Xs, ys = X[sub], y[sub]
    else:
        Xs, ys = X, y

    m = int(0.8 * len(ys))
    perm = rng.permutation(len(ys))
    tr, va = perm[:m], perm[m:]
    Xtr, ytr, Xval, yval = Xs[tr], ys[tr], Xs[va], ys[va]

    # Random-Forest importance prior
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed)
    rf.fit(Xtr, ytr)
    imp = rf.feature_importances_
    prior = (imp - imp.mean()) / (imp.std() + 1e-9)

    # chaotic (logistic map) initialization biased by the RF prior
    pos = np.zeros((cfg.swarm_size, D))
    c = rng.uniform(0.1, 0.9, size=D)
    for i in range(cfg.swarm_size):
        c = 4.0 * c * (1.0 - c)                     # logistic map, mu = 4
        pos[i] = 2 * (c - 0.5) + 0.5 * prior + rng.normal(0, 0.1, D)
    vel = rng.normal(0, 0.1, size=(cfg.swarm_size, D))

    pbest = pos.copy()
    pbest_f = np.array([_fitness(_topk_mask(p, cfg.k_features), Xtr, ytr, Xval, yval,
                                 cfg.lam_redundancy) for p in pos])
    g = pbest[pbest_f.argmax()].copy()
    gf = pbest_f.max()
    hist = [gf]

    for t in range(cfg.capso_iters):
        w = cfg.w_max - (cfg.w_max - cfg.w_min) * t / cfg.capso_iters   # time-varying inertia
        r1 = rng.random((cfg.swarm_size, D))
        r2 = rng.random((cfg.swarm_size, D))
        vel = w * vel + cfg.c1 * r1 * (pbest - pos) + cfg.c2 * r2 * (g[None, :] - pos)
        pos = pos + vel
        for i in range(cfg.swarm_size):
            f = _fitness(_topk_mask(pos[i], cfg.k_features), Xtr, ytr, Xval, yval,
                         cfg.lam_redundancy)
            if f > pbest_f[i]:
                pbest_f[i] = f
                pbest[i] = pos[i].copy()
        if pbest_f.max() > gf:
            gf = pbest_f.max()
            g = pbest[pbest_f.argmax()].copy()
        hist.append(gf)
        if verbose and (t % 10 == 0 or t == cfg.capso_iters - 1):
            print(f"[CAPSO] iter {t:>3d}  best-fitness {gf:.4f}")

    idx = _topk_mask(g, cfg.k_features)
    return np.sort(idx), np.array(hist)

## 9. Data re-uploading quantum layer (Section III-E)
6 qubits, depth D=3; angle encoding `RY(πx)RZ(πx)` re-uploaded each stage, trainable RY/RZ blocks, ring-CNOT entangler, ⟨Z⟩ read-out.

In [ ]:
"""Data re-uploading quantum layer (Section III-E) implemented in PennyLane."""
import subprocess
import sys
import numpy as np

try:
    import pennylane as qml
except ModuleNotFoundError:
    print("PennyLane is missing; installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pennylane"])
    import pennylane as qml


def build_quantum_layer(n_qubits=6, depth=3):
    """Return a torch-differentiable qml.qnn.TorchLayer for the re-uploading circuit.

    For each re-uploading stage d = 1..D:
      1. angle-encode every feature via R_Y(pi x_q) R_Z(pi x_q)   (Eq. 20)
      2. apply a trainable variational block R_Y(theta) R_Z(theta) (Eq. 21)
      3. entangle with a ring of CNOTs                            (Eq. 22)
    Read out <Z_q> on each qubit (Eq. 24).
    """
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        # inputs: (..., n_qubits) in [-1, 1]
        # weights: (depth, n_qubits, 2)
        for d in range(depth):
            for q in range(n_qubits):
                qml.RY(np.pi * inputs[..., q], wires=q)
                qml.RZ(np.pi * inputs[..., q], wires=q)

            for q in range(n_qubits):
                qml.RY(weights[d, q, 0], wires=q)
                qml.RZ(weights[d, q, 1], wires=q)

            for q in range(n_qubits):
                qml.CNOT(wires=[q, (q + 1) % n_qubits])

        return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

    weight_shapes = {"weights": (depth, n_qubits, 2)}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


## 10. Hybrid model: quantum layer + classical head (Section III-F)

In [ ]:
"""QDR-AttNet quantum layer + classical head (Sections III-E/F)."""
import torch
import torch.nn as nn



class QDRAttNetHead(nn.Module):
    """Operates on the k CAPSO-selected features: encode -> re-uploading circuit -> head."""

    def __init__(self, cfg=CFG):
        super().__init__()
        self.k = cfg.k_features
        self.q = build_quantum_layer(cfg.n_qubits, cfg.reupload_depth)
        self.head = nn.Sequential(
            nn.Linear(cfg.n_qubits + self.k, cfg.head_hidden),
            nn.BatchNorm1d(cfg.head_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.head_hidden, cfg.num_classes),
        )
        # standardization for the selected features (fit on the training fold)
        self.register_buffer("mean", torch.zeros(self.k))
        self.register_buffer("std", torch.ones(self.k))

    def set_scaler(self, mean, std):
        self.mean.copy_(torch.as_tensor(mean, dtype=torch.float32))
        self.std.copy_(torch.as_tensor(std, dtype=torch.float32) + 1e-6)

    def encode(self, x):
        return torch.tanh((x - self.mean) / self.std)     # bound features to (-1, 1)

    def forward(self, x, return_embed=False):
        xs = self.encode(x)
        q = self.q(xs)                                     # (B, n_qubits) quantum embedding
        logits = self.head(torch.cat([q, xs], dim=1))      # residual selected features
        return (logits, q) if return_embed else logits

## 11. Fifteen evaluation measures + calibration (ECE)

`compute_all_metrics` returns the fifteen macro-averaged measures of Table II. `expected_calibration_error` / `reliability_curve` add the calibration analysis of the *Confidence Calibration* subsection (Fig. 12). `compute_all_metrics_with_calibration` returns all sixteen together.

In [ ]:
"""Fifteen macro-averaged evaluation measures (Section IV-D)."""
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, jaccard_score,
    log_loss, hamming_loss, confusion_matrix,
)


def _specificity_macro(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    tot = cm.sum()
    spec = []
    for c in range(n_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        tn = tot - tp - fp - fn
        spec.append(tn / (tn + fp + 1e-12))
    return float(np.mean(spec))


def top_k_accuracy(y_true, y_prob, k=3):
    topk = np.argsort(-y_prob, axis=1)[:, :k]
    return float(np.mean([y_true[i] in topk[i] for i in range(len(y_true))]))


def compute_all_metrics(y_true, y_prob, n_classes):
    y_true = np.asarray(y_true)
    y_pred = y_prob.argmax(1)
    Y = np.eye(n_classes)[y_true]
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    spec = _specificity_macro(y_true, y_pred, n_classes)

    try:
        roc = roc_auc_score(Y, y_prob, average="macro", multi_class="ovr")
    except Exception:
        roc = float("nan")
    try:
        prc = average_precision_score(Y, y_prob, average="macro")
    except Exception:
        prc = float("nan")

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": rec,
        "specificity_macro": spec,
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "g_mean": float(np.sqrt(max(rec, 0.0) * max(spec, 0.0))),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc_macro": roc,
        "pr_auc_macro": prc,
        "jaccard_macro": jaccard_score(y_true, y_pred, average="macro", zero_division=0),
        "top3_accuracy": top_k_accuracy(y_true, y_prob, 3),
        "log_loss": log_loss(y_true, y_prob, labels=list(range(n_classes))),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }

# --------------------------------------------------------------------------
# Confidence calibration (paper: "Confidence Calibration" subsection, Fig. 12)
# ECE is reported separately from the fifteen ranking/accuracy metrics above.
# --------------------------------------------------------------------------
def expected_calibration_error(y_true, y_prob, n_bins=15):
    """Top-label Expected Calibration Error (equal-width confidence bins).

    ECE = sum_b (|B_b| / N) * | acc(B_b) - conf(B_b) |,
    where conf is the predicted probability of the top-1 class. Matches the
    reliability-diagram summary reported in the paper (lower is better).
    """
    y_true = np.asarray(y_true)
    conf = y_prob.max(1)
    pred = y_prob.argmax(1)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    N = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        # last bin is closed on the right so conf == 1.0 is counted
        in_bin = (conf > lo) & (conf <= hi) if hi < 1.0 else (conf > lo) & (conf <= hi + 1e-9)
        m = in_bin.sum()
        if m == 0:
            continue
        acc_bin = correct[in_bin].mean()
        conf_bin = conf[in_bin].mean()
        ece += (m / N) * abs(acc_bin - conf_bin)
    return float(ece)


def reliability_curve(y_true, y_prob, n_bins=15):
    """Return per-bin (mean_confidence, accuracy, fraction) for a reliability
    diagram (paper Fig. 12). Bins with no samples are returned as NaN."""
    y_true = np.asarray(y_true)
    conf = y_prob.max(1)
    correct = (y_prob.argmax(1) == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    mids, accs, fracs = [], [], []
    N = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (conf > lo) & (conf <= hi) if hi < 1.0 else (conf > lo) & (conf <= hi + 1e-9)
        m = in_bin.sum()
        mids.append((lo + hi) / 2.0)
        accs.append(float(correct[in_bin].mean()) if m else float("nan"))
        fracs.append(float(m / N))
    return np.array(mids), np.array(accs), np.array(fracs)


def compute_all_metrics_with_calibration(y_true, y_prob, n_classes, ece_bins=15):
    """Convenience wrapper: the fifteen macro metrics plus ECE (calibration)."""
    m = compute_all_metrics(y_true, y_prob, n_classes)
    m["ece"] = expected_calibration_error(y_true, y_prob, n_bins=ece_bins)
    return m


## 12. Statistical tests (Section V-F)

In [ ]:
"""Statistical significance utilities (Section V-F): paired t, Wilcoxon,
Friedman-Nemenyi, paired 95% confidence intervals, Cohen's d, and a
conservative fold-averaged (n=5) reanalysis."""
import numpy as np
from scipy import stats


def paired_ttest(a, b):
    t, p = stats.ttest_rel(a, b)
    return float(t), float(p)


def wilcoxon_test(a, b):
    try:
        w, p = stats.wilcoxon(a, b)
        return float(w), float(p)
    except ValueError:
        return float("nan"), 1.0


def cohens_d_paired(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    return float(d.mean() / (d.std(ddof=1) + 1e-12))


def paired_ci(a, b, conf=0.95):
    """Two-sided (1-conf) t confidence interval for the paired mean difference
    (a - b). Returns (low, high) on the same scale as the inputs. This backs
    the '95% CI (delta-pp)' column of Table VI."""
    d = np.asarray(a, float) - np.asarray(b, float)
    n = len(d)
    mean = d.mean()
    se = d.std(ddof=1) / np.sqrt(n) if n > 1 else 0.0
    tcrit = stats.t.ppf(0.5 + conf / 2.0, df=max(n - 1, 1))
    return float(mean - tcrit * se), float(mean + tcrit * se)


def friedman_test(*model_scores):
    stat, p = stats.friedmanchisquare(*model_scores)
    return float(stat), float(p)


def mean_ranks(score_matrix):
    """score_matrix: (n_folds, k_models), higher is better. Returns mean rank per model."""
    score_matrix = np.asarray(score_matrix, float)
    ranks = np.array([stats.rankdata(-row) for row in score_matrix])
    return ranks.mean(0)


def nemenyi_cd(k, n, alpha=0.05):
    """Critical difference for the Nemenyi post-hoc test (k models, n folds)."""
    q = {0.05: {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
                7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164},
         0.10: {2: 1.645, 3: 2.052, 4: 2.291, 5: 2.459, 6: 2.589,
                7: 2.693, 8: 2.780, 9: 2.855, 10: 2.920}}
    qa = q.get(alpha, q[0.05]).get(k, 3.164)
    return float(qa * np.sqrt(k * (k + 1) / (6.0 * n)))


def compare_to_baselines(proposed_f1, baseline_f1_dict, conf=0.95):
    """proposed_f1: array (n_obs,); baseline_f1_dict: {name: array (n_obs,)}.

    Reproduces Table VI: mean delta-F1 (pp), paired t, Wilcoxon, Cohen's d,
    and the paired 95% CI for delta-F1 (pp)."""
    rows = []
    for name, arr in baseline_f1_dict.items():
        t, pt = paired_ttest(proposed_f1, arr)
        _, pw = wilcoxon_test(proposed_f1, arr)
        d = cohens_d_paired(proposed_f1, arr)
        lo, hi = paired_ci(proposed_f1, arr, conf=conf)
        rows.append({
            "vs": name,
            "delta_f1_pp": 100.0 * (np.mean(proposed_f1) - np.mean(arr)),
            "t_stat": t, "paired_t_p": pt, "wilcoxon_p": pw, "cohens_d": d,
            "ci95_low_pp": 100.0 * lo, "ci95_high_pp": 100.0 * hi,
        })
    return rows


def fold_average(scores, folds, seeds):
    """Conservative fold-level reduction (revised Section V-F): average the
    per-seed scores within each fold to obtain one value per fold (n = n_folds),
    since seeds inside a fold share the same train/test partition and are not
    independent. `scores`, `folds`, `seeds` are equal-length sequences."""
    scores = np.asarray(scores, float)
    folds = np.asarray(folds)
    out = []
    for f in sorted(np.unique(folds)):
        out.append(scores[folds == f].mean())
    return np.array(out)


## 13. Two-phase training + cross-validation pipeline

In [ ]:
"""End-to-end training / evaluation for QDR-AttNet.

Pipeline (two phases, per fold):
  Phase A  : train QMI-attention front-end -> refine features -> CAPSO selects k=6 features
  Phase B  : train the data re-uploading quantum layer + classical head on the 6 features
Evaluation : 15 macro metrics + ECE (calibration) under 5-fold x 3-seed cross-validation.

Usage:
  python train.py --smoke --cpu          # verify the pipeline on synthetic features
  python extract_features.py             # cache backbone embeddings (needs the dataset)
  python train.py                        # full run on the cached embeddings
"""
import os
import json
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score



def fisher_sep(q, y):
    """Quantum-separability regularizer L_sep = tr(S_W)/tr(S_B) (Eq. 30)."""
    classes = torch.unique(y)
    g = q.mean(0)
    Sw = q.new_zeros(())
    Sb = q.new_zeros(())
    for c in classes:
        qc = q[y == c]
        mc = qc.mean(0)
        Sw = Sw + ((qc - mc) ** 2).sum()
        Sb = Sb + qc.shape[0] * ((mc - g) ** 2).sum()
    return Sw / (Sb + 1e-6)


@torch.no_grad()
def _f1_macro(model, X, y, device):
    preds = []
    for i in range(0, X.shape[0], 256):
        preds.append(model(X[i:i + 256].to(device)).argmax(1).cpu())
    return f1_score(y.cpu().numpy(), torch.cat(preds).numpy(), average="macro", zero_division=0)


@torch.no_grad()
def predict_proba(model, X, device):
    model.eval()
    out = []
    for i in range(0, X.shape[0], 256):
        xb = torch.as_tensor(X[i:i + 256], dtype=torch.float32, device=device)
        out.append(torch.softmax(model(xb), 1).cpu().numpy())
    return np.concatenate(out)


def train_head(model, Xtr, ytr, Xval, yval, cfg, device):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.as_tensor(ytr, dtype=torch.long)
    Xval_t = torch.as_tensor(Xval, dtype=torch.float32, device=device)
    yval_t = torch.as_tensor(yval, dtype=torch.long, device=device)
    best, best_state, bad, n = -1.0, None, 0, len(ytr_t)

    for ep in range(cfg.max_epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, cfg.batch_size):
            b = perm[i:i + cfg.batch_size]
            if len(b) < 2:      # BatchNorm needs >=2 samples
                continue
            xb = Xtr_t[b].to(device)
            yb = ytr_t[b].to(device)
            logits, q = model(xb, return_embed=True)
            loss = F.cross_entropy(logits, yb) + cfg.gamma_sep * fisher_sep(q, yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        vf = _f1_macro(model, Xval_t, yval_t, device)
        if vf > best:
            best, bad = vf, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def run_cv(u, v, y, cfg):
    device = get_device(cfg.device)
    ensure_dir(cfg.out_dir)
    per_run, f1_per_fold = [], []

    for seed in cfg.seeds:
        set_seed(seed)
        skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=seed)
        for fold, (tr_idx, te_idx) in enumerate(skf.split(u, y)):
            rng = np.random.default_rng(seed * 100 + fold)
            perm = rng.permutation(len(tr_idx))
            nval = max(2, int(cfg.val_frac * len(tr_idx)))
            va = tr_idx[perm[:nval]]
            tr = tr_idx[perm[nval:]]

            # Phase A: front-end distillation + CAPSO selection (on the training fold only)
            fe = train_frontend(u[tr], v[tr], y[tr], cfg, device, verbose=False)
            Z = compute_refined(fe, u, v, device)                       # (N, 1792)
            idx, _ = capso_select(Z[tr], y[tr], cfg, seed=seed, verbose=False)
            Xsel = Z[:, idx]

            # Phase B: quantum layer + head
            mean, std = Xsel[tr].mean(0), Xsel[tr].std(0)
            model = QDRAttNetHead(cfg)
            model.set_scaler(mean, std)
            model = train_head(model, Xsel[tr], y[tr], Xsel[va], y[va], cfg, device)

            proba = predict_proba(model, Xsel[te_idx], device)
            m = compute_all_metrics(y[te_idx], proba, cfg.num_classes)
            m["ece"] = expected_calibration_error(
                y[te_idx], proba, n_bins=getattr(cfg, "ece_bins", 15))   # calibration (Fig. 12)
            m.update({"seed": int(seed), "fold": int(fold), "selected": idx.tolist()})
            per_run.append(m)
            f1_per_fold.append(m["f1_macro"])
            print(f"seed {seed} fold {fold}: acc {m['accuracy']:.4f}  "
                  f"f1 {m['f1_macro']:.4f}  auc {m['roc_auc_macro']:.4f}  ece {m['ece']:.4f}")

    float_keys = [k for k, val in per_run[0].items() if isinstance(val, float)]
    agg = {k: [float(np.nanmean([r[k] for r in per_run])),
               float(np.nanstd([r[k] for r in per_run]))] for k in float_keys}

    json.dump({"per_run": per_run, "aggregate": agg, "f1_per_fold": f1_per_fold},
              open(os.path.join(cfg.out_dir, "results.json"), "w"), indent=2)

    print("\n=== Aggregate over folds (mean +/- std) ===")
    for k, (mu, sd) in agg.items():
        print(f"{k:20s} {mu:.4f} +/- {sd:.4f}")
    return agg, f1_per_fold


def smoke(cfg):
    """Run the whole pipeline on synthetic features (no dataset needed)."""
    print("SMOKE TEST: synthetic features, tiny settings\n")
    rng = np.random.default_rng(0)
    N = 400
    y = rng.integers(0, cfg.num_classes, N)
    u = rng.normal(size=(N, cfg.effnet_dim)).astype("float32")
    v = rng.normal(size=(N, cfg.convnext_dim)).astype("float32")
    for c in range(cfg.num_classes):                # inject a little class signal
        u[y == c, c % cfg.effnet_dim] += 3.0
        v[y == c, c % cfg.convnext_dim] += 3.0
    cfg.frontend_epochs = 3
    cfg.capso_iters = 6
    cfg.swarm_size = 8
    cfg.capso_subsample = 400
    cfg.max_epochs = 4
    cfg.n_folds = 2
    cfg.seeds = (42,)
    cfg.device = "cpu"
    run_cv(u, v, y, cfg)
    print("\nSMOKE TEST completed OK.")

## 14. Sanity check on synthetic data (no dataset needed)
Optional diagnostic only. The full Kaggle dataset has already been prepared near the top of the notebook.

In [ ]:
# OPTIONAL smoke test. Uncomment these lines only if you want a quick diagnostic.
# import copy
# cfg_smoke = copy.deepcopy(CFG)
# smoke(cfg_smoke)


## 15. Full run (on the real dataset)
The Kaggle dataset was prepared at the start of the notebook. This stage only validates that path, caches backbone features once, and runs cross-validation.

In [ ]:
# --- COMPLETE REAL-DATA RUN ---------------------------------------------------
# DATA_ROOT was downloaded/resolved near the TOP of this notebook.
# No dataset download occurs here.

if not is_valid_image_root(CFG.data_root, CFG.num_classes):
    raise FileNotFoundError(
        f"Configured dataset root is invalid: {CFG.data_root}. "
        "Rerun the dataset-preparation cell near the top of the notebook."
    )

print("Using prepared dataset:", CFG.data_root)
print("Number of classes:", len(_class_dirs(CFG.data_root)))

# 1) Extract frozen EfficientNetV2-S + ConvNeXt-Tiny embeddings once.
#    Subsequent runs reuse CFG.cache_path.
u, v, y = load_or_extract(
    root=CFG.data_root,
    batch_size=CFG.batch_size,
)

print("\nDataset ready")
print("Samples:", len(y))
print("EfficientNet features:", u.shape)
print("ConvNeXt features:", v.shape)
print("Unique classes:", len(np.unique(y)))

# 2) Run the complete QDR-AttNet cross-validation experiment.
agg, f1_per_fold = run_cv(u, v, y, CFG)

print("\nFinished. Results saved to:", os.path.join(CFG.out_dir, "results.json"))


Using prepared dataset: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Number of classes: 38
Feature extraction dataset: /kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train
Device: cpu
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:01<00:00, 50.9MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:01<00:00, 66.0MB/s]
extract features:  20%|██        | 221/1099 [1:57:33<7:43:26, 31.67s/it]

### Significance tests vs. baselines
Collect each competitor's per-fold macro-F1 (same splits/seeds) and compare. `compare_to_baselines` now also returns the paired **95% CI (Δpp)** column of Table VI, and `fold_average` gives the conservative fold-level (n=5) reanalysis described in Section V-F:
```python
baselines = {'P-HQNN': phqnn_f1, 'EfficientNet-B0': eff_f1}  # each: per-run macro-F1 (len = folds*seeds)
rows = compare_to_baselines(np.array(f1_per_fold), baselines)   # delta, t, Wilcoxon, Cohen's d, 95% CI

# conservative fold-level (n=5) check: average seeds within each fold first
folds = [r['fold'] for r in per_run]; seeds = [r['seed'] for r in per_run]
qdr_fold = fold_average(f1_per_fold, folds, seeds)
phqnn_fold = fold_average(phqnn_f1, folds, seeds)
rows_n5 = compare_to_baselines(qdr_fold, {'P-HQNN': phqnn_fold})
```
Use `friedman_test`, `mean_ranks`, `nemenyi_cd` for the multi-model ranking analysis. Mean **ECE** aggregates automatically in `run_cv` (printed per fold and stored in `results.json`).

**Reminder:** for hardware / shot-noise studies, switch the qnode in cell 8 to `diff_method='parameter-shift'` and use an SPSA optimizer.
